# Information model and estimator experiments

Various experiments involving the information models as well as the scores.

In [ ]:
# allow imports from the main source directory
import sys
sys.path.append('..')
import logging

import math
import itertools
import random
import logging
from functools import partial

import numpy as np

import matplotlib.pyplot as plt
from matplotlib import animation, rc

from IPython.display import display, HTML
from environment import SoilMoistureEnvironment, EpidemicSpreadEnvironment
from information_model import GaussianProcessScalarFieldIM, DiskEstimateScalarFieldIM, PointEstimateScalarFieldIM
from information_model import im_score, im_score_rmse, im_score_rmse_weighted

logging.basicConfig(level=logging.WARNING)

In [ ]:
def visualize_env_im(env, im):
    """Visualize both the environment and the information model"""
    fig, (axenv, aximval, aximunc) = plt.subplots(1, 3, figsize = (12, 4))
    axenv.imshow(env.value, vmin=0, vmax=1.0, cmap="gray")
    axenv.set_title("Environment")
    aximval.imshow(im.value, vmin=0, vmax=1.0, cmap="gray")
    aximval.set_title("IM value")
    aximunc.imshow(im.uncertainty, vmin=0, vmax=1.0, cmap="gray")
    aximunc.set_title("IM uncertainty")


## Example 1: soil moisture environment

In [ ]:
# create an environment to observe
env = SoilMoistureEnvironment("water", width = 100, height = 100, seed=1)
plt.imshow(env.value, vmin=0, vmax=1.0, cmap="gray")

In [ ]:
# generate a series random observations
observations = []
for i in range(100):
    x = random.randint(0, env.width-1)
    y = random.randint(0, env.height-1)
    value = env.value[x,y]
    obs = {"x": x, "y": y, "value": value}
    observations.append(obs)

In [ ]:
im = GaussianProcessScalarFieldIM(env.width, env.height)
for obs in observations:
    im.add_observation(obs)
im.proceed(1)
visualize_env_im(env, im)
score_rmse = im_score_rmse(im, env)
print(f"Score_rmse = {score_rmse}")

In [ ]:
im = DiskEstimateScalarFieldIM(env.width, env.height, disk_radius=None, default_value=0.5)
for obs in observations:
    im.add_observation(obs)
im.proceed(1)
visualize_env_im(env, im)
score_rmse = im_score_rmse(im, env)
print(f"Score_rmse = {score_rmse}")

In [ ]:
mask = np.ones([100, 100])
score_rmse_weighted = im_score_rmse_weighted(im, env, mask)
print(f"Score_rmse_weighted = {score_rmse_weighted}")

# Experiments with the epidemic spread environment

In [ ]:
# Epidemic spread with immunity mask
width = 10 # 1000
height = 10 # 2000
immunity_mask = np.zeros((width, height))
for i in range(width):
    for j in range(2):
        immunity_mask[i, height // 2 + j] = -2.0

#spread_dimension = int(math.sqrt(width) / 6) * 2 + 3
spread_dimension = 11
print(spread_dimension)
infection_seeds = 1 # 3 * max(int(width / 30), 1)
p_transmission = 0.25
# seeds 1*, 0.15 looks good for ccr
# seeds 3*, 0.25 looks good for tylcv
env = EpidemicSpreadEnvironment("crop", width, height, seed=1, infection_duration = 5, 
                                p_transmission = p_transmission, spread_dimension=spread_dimension, 
                                infection_seeds = infection_seeds, immunity_mask=immunity_mask)
for t in range(10):
    env.proceed(1.0)

In [ ]:
plt.imshow(env.value, vmin=0, vmax=1.0, cmap="gray")

In [ ]:
def estimate_with_disk(env, observations): 
    im = DiskEstimateScalarFieldIM(env.width, env.height, disk_radius=None, default_value=0.5)
    for obs in observations:
        im.add_observation(obs)
    im.proceed(1)
    return im

#visualize_env_im(env, im)
#score_rmse = im_score_rmse(im, env)
#print(f"Score_rmse = {score_rmse}")

In [ ]:
def estimate_with_GP(env, observations):
    im = GaussianProcessScalarFieldIM(env.width, env.height)
    for obs in observations:
        im.add_observation(obs)
    im.proceed(1)
    return im

#    visualize_env_im(env, im)
#    score_rmse = im_score_rmse(im, env)
#    print(f"Score_rmse = {score_rmse}")


In [ ]:
# generate a series random observations
observations = []
ims_gp = []
ims_disk = []
scores_gp = []
scores_disk = []
for i in range(100):
    print(i)
    x = random.randint(0, env.width-1)
    y = random.randint(0, env.height-1)
    value = env.value[x,y]
    obs = {"x": x, "y": y, "value": value}
    observations.append(obs)
    # estimate with disk and score it
    im_disk = estimate_with_disk(env, observations)
    ims_disk.append(im_disk)
    score_disk = im_score_rmse(im_disk, env)
    scores_disk.append(score_disk)
    # estimate with GP and score it
    im_gp = estimate_with_GP(env, observations)
    ims_gp.append(im_gp)
    score_gp = im_score_rmse(im_gp, env)
    scores_gp.append(score_gp)

In [ ]:
# plot the results
fig, ax = plt.subplots(1, figsize=(5,5))
ax.plot(scores_gp, label="score GP")
ax.plot(scores_disk, label="score Disk")
ax.set_ylim(top=0)
ax.set_xlabel("Time")
ax.legend()

In [ ]:
def show_difference(env, im):
    difference = env.value - im.value
    fig, (axenv, aximval, axdiff) = plt.subplots(1, 3, figsize = (12, 4))
    axenv.imshow(env.value, vmin=0, vmax=1.0, cmap="gray")
    axenv.set_title("Environment")
    aximval.imshow(im.value, vmin=0, vmax=1.0, cmap="gray")
    aximval.set_title("IM value")
    axdiff.imshow(difference, vmin=-1.0, vmax=1.0, cmap="bwr")
    axdiff.set_title("difference")

In [ ]:
show_difference(env, im_gp)

In [ ]:
show_difference(env, im_disk)

In [ ]:
def show_mask(env, immunity_mask):
    fig, ax = plt.subplots(1, 1, figsize = (4, 4))
    ax.imshow(env.value, vmin=0, vmax=1.0, cmap="gray")
    immunity_val = -immunity_mask / 2.0
    ax.imshow(immunity_val, vmin=-1.0, vmax=1.0, alpha=0.1, cmap="bwr")

show_mask(env, immunity_mask)

In [ ]:
immunity_mask